In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append('../../..')

In [ ]:
from PitchModel.Stuff.Tuning.TuningData import TuningData

tuning_data = TuningData(start_year=2024, end_year=2024)

In [ ]:
from PitchModel.Stuff.Tuning.TuneFunction import TuneGroup, GetStudyName, GetTunedParameterNames
from PitchModel.Stuff.Tuning.TunePrepMapFunction import RunPrepMapEvaluation, GetPrepMapGrid, EnqueuePrepMapGrid, SummarizePrepMapStudy
from PitchModel.Stuff.Model.ModelOutputType import ModelVariantType, ModelOutputType
from functools import partial

variant_type = ModelVariantType.Combined
output_type = ModelOutputType.SwingResults
num_repeats = 3
evaluations_per_config = 2

# Only SpinAxis, ZoneHeight (Combined only), PcaDrop
tune_groups = [TuneGroup.SpinAxis, TuneGroup.ZoneHeight, TuneGroup.PcaDrop]

objective_function = partial(
    RunPrepMapEvaluation,
    tuning_data=tuning_data,
    model_variant_type=variant_type,
    model_output_type=output_type,
    max_repeats=num_repeats,
    groups=tune_groups
)

In [ ]:
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)
study_name = GetStudyName(variant_type, output_type, tune_groups, tag="Grid")
#optuna.delete_study(study_name=study_name, storage="sqlite:///Tune.db")
study = optuna.create_study(
    direction="minimize",
    load_if_exists=True,
    study_name=study_name,
    storage="sqlite:///Tune.db"
)

n_trials = EnqueuePrepMapGrid(study, tune_groups, evaluations_per_config)
print(f"{len(GetPrepMapGrid(tune_groups))} PrepMap configs, {n_trials} trials waiting to run")

In [ ]:
study.optimize(objective_function, n_trials=n_trials, show_progress_bar=True)

In [ ]:
summary = SummarizePrepMapStudy(study, tune_groups)
summary

In [ ]:
# Average of each option across all values of the other options
loss_columns = [c for c in ["mean_loss", "delta_vs_base"] if c in summary.columns]
for name in GetTunedParameterNames(tune_groups):
    display(summary.groupby(name)[loss_columns].mean().sort_values("mean_loss"))